# Advanced Pipeline Inspection Demo

This notebook demonstrates how to run a multi-stage processing pipeline using in-memory configuration, bypassing the need for external YAML files. We focus on using `get_data()` to inspect data after specific QC milestones.

The data used is the Nelson (unit_397) glider dataset from the BOCD Bio-Carbon Deployment Catalogue.

In [1]:
# This file is part of the NOC Autonomy Toolbox.
#
# Copyright 2025-2026 National Oceanography Centre and The Contributors
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#    http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

In [2]:
import os
import sys
import numpy as np
from pathlib import Path
import requests

# --- Friendly Configuration Variables ---
DATA_URL = "https://linkedsystems.uk/erddap/files/Public_OG1_Data_001/Nelson_20240528/Nelson_646_R.nc"
DATA_DIR = Path("../data/OG1")
INPUT_FILE = DATA_DIR / "Nelson_646_R.nc"
TOOLBOX_PATH = "../../src"
OUTPUT_DIRECTORY = "./"

# Set working directory and download data
if not DATA_DIR.exists():
    DATA_DIR.mkdir(parents=True)

if not INPUT_FILE.exists():
    response = requests.get(DATA_URL)
    if response.status_code == 200:
        with open(INPUT_FILE, "wb") as f:
            f.write(response.content)
        print(f"Data downloaded to {INPUT_FILE}")
    else:
        print("Data download failed")

# Import toolbox
sys.path.append(os.path.abspath(TOOLBOX_PATH))
from toolbox.pipeline import Pipeline, _setup_logging

[Discovery] Scanning for step modules in /Users/orlpru/Documents/Github/pelagos-py/src/toolbox/steps/custom
[Discovery] Importing step module: toolbox.steps.custom.find_profiles_beta
[Discovery] Importing step module: toolbox.steps.custom.blank_step
[Discovery] Importing step module: toolbox.steps.custom.apply_qc
[Discovery] Importing step module: toolbox.steps.custom.derive_ctd
[Discovery] Importing step module: toolbox.steps.custom.find_profiles
[Discovery] Importing step module: toolbox.steps.custom.export
[Discovery] Importing step module: toolbox.steps.custom.gen_data
[Discovery] Importing step module: toolbox.steps.custom.profile_direction
[Discovery] Importing step module: toolbox.steps.custom.interpolate_data
[Discovery] Importing step module: toolbox.steps.custom.load_data
[Discovery] Importing step module: toolbox.steps.custom.write_report
[Discovery] Importing step module: toolbox.steps.custom.variables.salinity
[Discovery] Importing step module: toolbox.steps.custom.variabl

## Virtual Pipeline Setup

Instead of loading from a file, we initialise an empty pipeline and inject the configuration directly. We also initialise logging to the console only, ensuring no log files are generated on the laptop.

In [3]:
# Define configuration in memory
config = {
    "pipeline": {
        "name": "Virtual Inspection Pipeline",
        "out_directory": OUTPUT_DIRECTORY,
        "visualisation": False
    },
    "steps": [
        {
            "name": "Load OG1",
            "parameters": {"file_path": str(INPUT_FILE.resolve())},
            "diagnostics": False
        }
    ]
}

# Initialise pipeline without a file path
pipe = Pipeline()

# Manually set parameters and logging (no log file specified)
pipe.global_parameters = config["pipeline"]
pipe.logger = _setup_logging(out_dir=OUTPUT_DIRECTORY, log_file=None)
pipe.build_steps(config["steps"])

# Run the initial load step
pipe.run()
print("Initial data loaded.")
data_after_load = pipe.get_data()
data_after_load

2026-04-09 10:36:47 - INFO - toolbox.pipeline - Assembling steps to run from config.
2026-04-09 10:36:47 - INFO - toolbox.pipeline - Step 'Load OG1' added successfully!
2026-04-09 10:36:47 - INFO - toolbox.pipeline - Executing: Load OG1
2026-04-09 10:36:48 - INFO - toolbox.pipeline.step.Load OG1 - [Load OG1] Loaded data from /Users/orlpru/Documents/Github/pelagos-py/examples/data/OG1/Nelson_646_R.nc
2026-04-09 10:36:48 - INFO - toolbox.pipeline.step.Load OG1 - [Load OG1] Loading dataset to RAM...
2026-04-09 10:36:48 - WARNING - toolbox.pipeline.step.Load OG1 - [Load OG1] Removed 672 records containing invalid or pre-deployment timestamps.


Initial data loaded.


[Load OG1] WARNING: Removed 672 records containing invalid or pre-deployment timestamps.


<xarray.Dataset> Size: 443MB
Dimensions:                               (N_MEASUREMENTS: 2014184, N_PARAM: 24)
Dimensions without coordinates: N_MEASUREMENTS, N_PARAM
Data variables: (12/68)
    TIME                                  (N_MEASUREMENTS) datetime64[ns] 16MB ...
    TIME_GPS                              (N_MEASUREMENTS) datetime64[ns] 16MB ...
    PHASE                                 (N_MEASUREMENTS) float32 8MB nan .....
    PHASE_QC                              (N_MEASUREMENTS) float32 8MB nan .....
    CNDC                                  (N_MEASUREMENTS) float32 8MB -0.001...
    PRES                                  (N_MEASUREMENTS) float32 8MB -0.17 ...
    ...                                    ...
    PLATFORM_TYPE                         <U6 24B 'slocum'
    PLATFORM_MODEL                        <U2 8B 'G2'
    WMO_IDENTIFIER                        <U3 12B '830'
    DEPLOYMENT_TIME                       datetime64[ns] 8B 2024-05-27T23:00:00
    DEPLOYMENT_LATITUDE                   <U17 68B '50.89280319213867'
    DEPLOYMENT_LONGITUDE                  <U19 76B '-1.3955066204071045'
Attributes: (12/68)
    geospatial_bounds_crs:           EPSG:4326
    geospatial_bounds_vertical_crs:  EPSG:5831
    geospatial_lat_min:              50.890984
    geospatial_lat_max:              62.482586
    geospatial_lon_min:              50.890984
    geospatial_lon_max:              62.482586
    ...                              ...
    instrument:                      ['SBE Slocum Glider Payload (GPCTD) CTD'...
    metadata_link:                   https://api.linked-systems.uk/api/meta/v...
    trajectory:                      Nelson_20240528
    date_created:                    2024-09-30T08:14:20.465997
    date_modified:                   2024-09-30T08:14:20.466021
    id:                              Nelson_20240528T000000_R

## Coordinate Quality Control

We now add and run the first major QC block which focuses on TIME, LATITUDE, and LONGITUDE. After running, we use `get_data()` to inspect the dataset state.

In [4]:
pipe.add_step(
    step_name="Apply QC",
    parameters={
        "qc_settings": {
            "impossible date qc": {},
            "impossible location qc": {},
            "position on land qc": {},
            "impossible speed qc": {}
        }
    },
    diagnostics=False,
    run_immediately=True
)

# Inspect data after Coordinate QC
data_after_coords = pipe.get_data()
data_after_coords

2026-04-09 10:36:48 - INFO - toolbox.pipeline - Step 'Apply QC' added successfully!
2026-04-09 10:36:48 - INFO - toolbox.pipeline - Running step 'Apply QC' immediately.
2026-04-09 10:36:48 - INFO - toolbox.pipeline - Executing: Apply QC
2026-04-09 10:36:48 - INFO - toolbox.pipeline.step.Apply QC - [Apply QC] Data found in context.
2026-04-09 10:36:48 - INFO - toolbox.pipeline.step.Apply QC - [Apply QC] Found existing flags columns {'LATITUDE_QC', 'LONGITUDE_QC'} in data.
2026-04-09 10:36:49 - WARNING - toolbox.pipeline.step.Apply QC - [Apply QC] Sanitised invalid flags in LATITUDE_QC: found NaN values.
[Apply QC] WARNING: Sanitised invalid flags in LATITUDE_QC: found NaN values.
2026-04-09 10:36:49 - WARNING - toolbox.pipeline.step.Apply QC - [Apply QC] Sanitised invalid flags in LONGITUDE_QC: found NaN values.
[Apply QC] WARNING: Sanitised invalid flags in LONGITUDE_QC: found NaN values.
2026-04-09 10:36:49 - INFO - toolbox.pipeline.step.Apply QC - [Apply QC] Found QC columns for unte

<xarray.Dataset> Size: 433MB
Dimensions:                               (N_MEASUREMENTS: 2014184, N_PARAM: 24)
Dimensions without coordinates: N_MEASUREMENTS, N_PARAM
Data variables: (12/69)
    TIME                                  (N_MEASUREMENTS) datetime64[ns] 16MB ...
    TIME_GPS                              (N_MEASUREMENTS) datetime64[ns] 16MB ...
    PHASE                                 (N_MEASUREMENTS) float32 8MB nan .....
    PHASE_QC                              (N_MEASUREMENTS) float32 8MB nan .....
    CNDC                                  (N_MEASUREMENTS) float32 8MB -0.001...
    PRES                                  (N_MEASUREMENTS) float32 8MB -0.17 ...
    ...                                    ...
    PLATFORM_MODEL                        <U2 8B 'G2'
    WMO_IDENTIFIER                        <U3 12B '830'
    DEPLOYMENT_TIME                       datetime64[ns] 8B 2024-05-27T23:00:00
    DEPLOYMENT_LATITUDE                   <U17 68B '50.89280319213867'
    DEPLOYMENT_LONGITUDE                  <U19 76B '-1.3955066204071045'
    TIME_QC                               (N_MEASUREMENTS) int8 2MB 1 1 ... 1 1
Attributes: (12/68)
    geospatial_bounds_crs:           EPSG:4326
    geospatial_bounds_vertical_crs:  EPSG:5831
    geospatial_lat_min:              50.890984
    geospatial_lat_max:              62.482586
    geospatial_lon_min:              50.890984
    geospatial_lon_max:              62.482586
    ...                              ...
    instrument:                      ['SBE Slocum Glider Payload (GPCTD) CTD'...
    metadata_link:                   https://api.linked-systems.uk/api/meta/v...
    trajectory:                      Nelson_20240528
    date_created:                    2024-09-30T08:14:20.465997
    date_modified:                   2024-09-30T08:14:20.466021
    id:                              Nelson_20240528T000000_R

## CTD Quality Control

Next, we apply complex QC to the CTD variables. This includes range checks and stuck value detection for Pressure (PRES), with flags being propagated to Conductivity (CNDC) and Temperature (TEMP).

In [5]:
pipe.add_step(
    step_name="Apply QC",
    parameters={
        "qc_settings": {
            "impossible range qc": {
                "variable_ranges": {
                    "PRES": {
                        3: [-5, -2.4],
                        4: [float("-inf"), -5]
                    }
                },
                "also_flag": {"PRES": ["CNDC", "TEMP"]},
                "plot": ["PRES"]
            },
            "stuck value qc": {
                "variables": {"PRES": 2},
                "also_flag": {"PRES": ["CNDC", "TEMP"]},
                "plot": ["PRES"]
            }
        }
    },
    diagnostics=False,
    run_immediately=True
)

# Final data extraction for the next inspection
final_data = pipe.get_data()
final_data

2026-04-09 10:36:50 - INFO - toolbox.pipeline - Step 'Apply QC' added successfully!
2026-04-09 10:36:50 - INFO - toolbox.pipeline - Running step 'Apply QC' immediately.
2026-04-09 10:36:50 - INFO - toolbox.pipeline - Executing: Apply QC
2026-04-09 10:36:50 - INFO - toolbox.pipeline.step.Apply QC - [Apply QC] Data found in context.
2026-04-09 10:36:50 - INFO - toolbox.pipeline.step.Apply QC - [Apply QC] Found existing flags columns {'PRES_QC', 'TEMP_QC', 'CNDC_QC'} in data.
2026-04-09 10:36:50 - WARNING - toolbox.pipeline.step.Apply QC - [Apply QC] Sanitised invalid flags in CNDC_QC: found NaN values.
[Apply QC] WARNING: Sanitised invalid flags in CNDC_QC: found NaN values.
2026-04-09 10:36:51 - WARNING - toolbox.pipeline.step.Apply QC - [Apply QC] Sanitised invalid flags in PRES_QC: found NaN values.
[Apply QC] WARNING: Sanitised invalid flags in PRES_QC: found NaN values.
2026-04-09 10:36:52 - WARNING - toolbox.pipeline.step.Apply QC - [Apply QC] Sanitised invalid flags in TEMP_QC: fo

<xarray.Dataset> Size: 415MB
Dimensions:                               (N_MEASUREMENTS: 2014184, N_PARAM: 24)
Dimensions without coordinates: N_MEASUREMENTS, N_PARAM
Data variables: (12/69)
    TIME                                  (N_MEASUREMENTS) datetime64[ns] 16MB ...
    TIME_GPS                              (N_MEASUREMENTS) datetime64[ns] 16MB ...
    PHASE                                 (N_MEASUREMENTS) float32 8MB nan .....
    PHASE_QC                              (N_MEASUREMENTS) float32 8MB nan .....
    CNDC                                  (N_MEASUREMENTS) float32 8MB -0.001...
    PRES                                  (N_MEASUREMENTS) float32 8MB -0.17 ...
    ...                                    ...
    PLATFORM_MODEL                        <U2 8B 'G2'
    WMO_IDENTIFIER                        <U3 12B '830'
    DEPLOYMENT_TIME                       datetime64[ns] 8B 2024-05-27T23:00:00
    DEPLOYMENT_LATITUDE                   <U17 68B '50.89280319213867'
    DEPLOYMENT_LONGITUDE                  <U19 76B '-1.3955066204071045'
    TIME_QC                               (N_MEASUREMENTS) int8 2MB 1 1 ... 1 1
Attributes: (12/68)
    geospatial_bounds_crs:           EPSG:4326
    geospatial_bounds_vertical_crs:  EPSG:5831
    geospatial_lat_min:              50.890984
    geospatial_lat_max:              62.482586
    geospatial_lon_min:              50.890984
    geospatial_lon_max:              62.482586
    ...                              ...
    instrument:                      ['SBE Slocum Glider Payload (GPCTD) CTD'...
    metadata_link:                   https://api.linked-systems.uk/api/meta/v...
    trajectory:                      Nelson_20240528
    date_created:                    2024-09-30T08:14:20.465997
    date_modified:                   2024-09-30T08:14:20.466021
    id:                              Nelson_20240528T000000_R

## Automated Flag Validation

One major advantage of `get_data()` is the ability to run custom validation scripts part-way through or at the end of a process. Below, we loop through our core measurement variables to ensure that a corresponding `_QC` variable exists and that every data point has been assigned a valid ARGO flag (0 to 9), ensuring no missing (NaN) flags remain.

In [ ]:
# --- Validation Configuration ---
data = pipe.get_data()

# We only want to check variables that are measurements (have N_MEASUREMENTS dimension)
# Metadata scalars don't need QC columns
measurement_vars = [v for v in data.data_vars if "N_MEASUREMENTS" in data[v].dims and not v.endswith("_QC")]

print(f"Starting Global QC Flag Validation for {len(measurement_vars)} measurement variables...\n")

for var in measurement_vars:
    qc_var = f"{var}_QC"
    
    # 1. Check for existence
    if qc_var not in data.data_vars:
        print(f"[FAILURE] {qc_var} is missing from the dataset.")
        continue
    
    qc_values = data[qc_var].values
    
    # 2. Check for NaNs first
    if np.issubdtype(qc_values.dtype, np.floating) and np.isnan(qc_values).any():
        print(f"[FAILURE] {qc_var} contains bad values!")
    
    # 3. If no NaNs, check the range
    else:
        valid_mask = (qc_values >= 0) & (qc_values <= 9)
        if valid_mask.all():
            print(f"[SUCCESS] {qc_var} is valid (0-9).")
        else:
            invalid_flags = np.unique(qc_values[~valid_mask])
            print(f"[FAILURE] {qc_var} contains invalid flags: {invalid_flags}")

Starting Global QC Flag Validation for 27 measurement variables...

[SUCCESS] TIME_QC is valid (0-9).
[SUCCESS] TIME_GPS_QC is valid (0-9).
[FAILURE] PHASE_QC contains NaN values!!!
[SUCCESS] CNDC_QC is valid (0-9).
[SUCCESS] PRES_QC is valid (0-9).
[SUCCESS] TEMP_QC is valid (0-9).
[FAILURE] CHLA_QC contains NaN values!!!
[FAILURE] BBP700_QC contains NaN values!!!
[FAILURE] BBP532_QC contains NaN values!!!
[FAILURE] OXYCOPVR_QC contains NaN values!!!
[FAILURE] BPHASE_DOXY_QC contains NaN values!!!
[FAILURE] OXYCOPVB_QC contains NaN values!!!
[FAILURE] OXYSAT_DOXY_QC contains NaN values!!!
[FAILURE] DPHASE_DOXY_QC contains NaN values!!!
[FAILURE] MOLAR_DOXY_QC contains NaN values!!!
[FAILURE] RAW_DOWNWELLING_PAR_QC contains NaN values!!!
[FAILURE] DOWNWELLING_PAR_QC contains NaN values!!!
[FAILURE] LATITUDE_GPS_QC contains NaN values!!!
[FAILURE] LONGITUDE_GPS_QC contains NaN values!!!
[FAILURE] GLIDER_DEPTH_QC contains NaN values!!!
[FAILURE] WATERCURRENTS_U_QC contains NaN values!!!


## Summary

By using `get_data()` at each milestone, we verified the addition of QC flags without needing to reload the dataset or write intermediate files to disk. The validation step above confirms that our logic correctly handles flags within the ARGO compliant 0-9 range.